In [1]:
import os
import random
from utils.hand_model_lite import HandModelMJCFLite 
 
import numpy as np
import transforms3d
import torch
import trimesh
import json
import plotly.graph_objects as go


/opt/conda/envs/dexgraspnet/lib/python3.7/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Here you need to choose your `hand_name`

In [2]:
mesh_path = "../data/meshdata"
# hand_name ="shadow_dexee"
hand_name ="DIP-Flex_opened_kinematics"

if hand_name == "shadow_dexee":
    ''' For Shadow Hand'''
    data_path = "../data/dataset/shadow_dexee/"
    hand_file = "mjcf/shadow_dexee/shadow_dexee.xml"
    joint_names = [
                    "F0_J0", "F0_J1", "F0_J2", "F0_J3", "F1_J0", "F1_J1", "F1_J2", "F1_J3", "F2_J0", "F2_J1", "F2_J2", "F2_J3"
    ]

elif hand_name =="barret":
    ''' For BarretHand'''
    data_path = "../data/dataset/barret/"
    hand_file = "mjcf/barret/barret.xml"
    joint_names = [
                    "wam_bhand_finger_1_prox_joint", "wam_bhand_finger_1_med_joint", "wam_bhand_finger_1_dist_joint", 
                    "wam_bhand_finger_2_prox_joint", "wam_bhand_finger_2_med_joint", "wam_bhand_finger_2_dist_joint",
                    "wam_bhand_finger_3_med_joint", "wam_bhand_finger_3_dist_joint"
    ]

elif hand_name =="DIP-Flex_opened_kinematics":
    ''' For Egorhand'''
    data_path = "../data/dataset/DIP-Flex_opened_kinematics"
    hand_file = "mjcf/DIP-Flex_opened_kinematics/DIP-Flex_opened_kinematics.xml"
    joint_names = [
                    "Joint_pinkie_abduction", "Joint_pinkie_PPflexion", "Joint_pinkie_DPflexion",
                    "Joint_index_abduction", "Joint_index_PPflexion", "Joint_index_DPflexion",
                    "Joint_thumb_rotation", "Joint_thumb_abduction", "Joint_thumb_PPflexion", "Joint_thumb_DPflexion"
    ]

elif hand_name == "robotiq_2":
    '''For robotiq'''
    data_path = "../data/dataset/robotiq_2/"
    hand_file = "mjcf/robotiq_2/robotiq_2 simpl.xml"
    joint_names = [
                    "left_spring_link_joint", "left_follower",
                    "right_spring_link_joint", "right_follower_joint"
    ]

elif hand_name == "panda":
    '''For panda'''
    data_path = "../data/dataset/panda/"
    hand_file = "mjcf/panda/panda.xml"
    joint_names = [
                    "finger_joint1", "finger_joint2"
    ]

elif hand_name == "shadow_dex_ee_simpl":
    data_path = "../data/dataset/shadow_dexee/"
    hand_file = "mjcf/shadow_dexee/shadow_dexee simpl.xml"
    joint_names = [
                    "F0_J0", "F0_J1", "F0_J2", "F0_J3", "F1_J0", "F1_J1", "F1_J2", "F1_J3", "F2_J0", "F2_J1", "F2_J2", "F2_J3"
    ]
elif hand_name =="barret_simpl":
    ''' For BarretHand'''
    data_path = "../data/dataset/barret/"
    hand_file = "mjcf/barret/barret_simpl.xml"
    joint_names = [
                    "wam_bhand_finger_1_prox_joint", "wam_bhand_finger_1_med_joint", "wam_bhand_finger_1_dist_joint", 
                    "wam_bhand_finger_2_prox_joint", "wam_bhand_finger_2_med_joint", "wam_bhand_finger_2_dist_joint",
                    "wam_bhand_finger_3_med_joint", "wam_bhand_finger_3_dist_joint"
    ]

elif hand_name =="DIP-Flex_opened_kinematics_simpl":
    ''' For Egorhand'''
    data_path = "../data/dataset/DIP-Flex_opened_kinematics"
    hand_file = "mjcf/DIP-Flex_opened_kinematics/DIP-Flex_opened_kinematics_simpl.xml"
    joint_names = [
                    "Joint_pinkie_abduction", "Joint_pinkie_PPflexion", "Joint_pinkie_DPflexion",
                    "Joint_index_abduction", "Joint_index_PPflexion", "Joint_index_DPflexion",
                    "Joint_thumb_rotation", "Joint_thumb_abduction", "Joint_thumb_PPflexion", "Joint_thumb_DPflexion"
    ]
 
    

translation_names = ['WRJTx', 'WRJTy', 'WRJTz']
rot_names = ['WRJRx', 'WRJRy', 'WRJRz']

In [3]:
hand_config = json.load(open('mjcf/' + hand_name + '/' + hand_name + '.json', 'r'))
device = "cpu"

In [4]:
hand_model = HandModelMJCFLite(
    hand_file,
    "mjcf/" + "assets/" + hand_name )

In [5]:
grasp_code_list = []
for code in os.listdir(data_path):
    grasp_code_list.append(code[:-4])

print(grasp_code_list)

['ddg-gd_banana_poisson_002', 'core-mug-8570d9a8d24cb0acbebd3c0c0c70fb03', 'sem-Bottle-437678d4bc6be981c8724d5673a063a6', 'mujoco-Ecoforms_Plant_Plate_S11Turquoise', 'hummer', 'sem-Camera-7bff4fd4dc53de7496dece3f86cb5dd5', 'pliers', 'screwdriver']


In [6]:
grasp_code = random.choice(grasp_code_list)
grasp_data = np.load(
    os.path.join(data_path, grasp_code+".npy"), allow_pickle=True)
object_mesh_origin = trimesh.load(os.path.join(
    mesh_path, grasp_code, "coacd/decomposed.obj"))
print(grasp_code)

print(grasp_data)
print(len(grasp_data))

ddg-gd_banana_poisson_002
[{'scale': 0.10000000149011612, 'qpos': {'Joint_pinkie_abduction': 0.06134083494544029, 'Joint_pinkie_PPflexion': 0.0009747938020154834, 'Joint_pinkie_DPflexion': 0.000477201072499156, 'Joint_index_abduction': 0.008731546811759472, 'Joint_index_PPflexion': 0.3319261372089386, 'Joint_index_DPflexion': 0.46106255054473877, 'Joint_thumb_rotation': 0.041159097105264664, 'Joint_thumb_abduction': 1.5236212015151978, 'Joint_thumb_PPflexion': 0.00023768239771015942, 'Joint_thumb_DPflexion': 0.2644842565059662, 'WRJRx': 2.278434599489423, 'WRJRy': 0.5318600953014201, 'WRJRz': 1.3295638859131031, 'WRJTx': -0.019685864448547363, 'WRJTy': 0.10977824032306671, 'WRJTz': 0.12781278789043427}, 'qpos_st': {'Joint_pinkie_abduction': 0.1333974450826645, 'Joint_pinkie_PPflexion': 0.152864009141922, 'Joint_pinkie_DPflexion': 0.038503240793943405, 'Joint_index_abduction': 0.07480882853269577, 'Joint_index_PPflexion': 0.20998017489910126, 'Joint_index_DPflexion': 0.4655381143093109,

In [7]:
grasp_obj = "sem-Hammer-405f308492a6f40d2c3380317c2cc450"
mesh_path_selected = "../data/selected_mesh"

In [8]:


index = random.randint(0, len(grasp_data) - 1)
# index = 3

qpos = grasp_data[index]['qpos']
print(index)
rot = np.array(transforms3d.euler.euler2mat(
    *[qpos[name] for name in rot_names]))
rot = rot[:, :2].T.ravel().tolist()
hand_pose = torch.tensor([qpos[name] for name in translation_names] + rot + [qpos[name]
                         for name in joint_names], dtype=torch.float, device="cpu").unsqueeze(0)
hand_model.set_parameters(hand_pose)
hand_mesh = hand_model.get_trimesh_data(0)
object_mesh = object_mesh_origin.copy().apply_scale(grasp_data[index]["scale"])

# Задаем цвета (RGB в формате [R, G, B, A], где значения от 0.0 до 1.0)
hand_color = [0.7, 0.7, 0.7, 1.0]  # Красноватый цвет для руки
object_color = [0.2, 0.5, 0.8, 1.0]  # Голубоватый цвет для объекта

# Применяем цвета к мешам
hand_mesh.visual.face_colors = hand_color
object_mesh.visual.face_colors = object_color




6


In [9]:
(hand_mesh+object_mesh).show()
